In [ ]:
!pip install -q fastapi uvicorn pyngrok nest-asyncio transformers bitsandbytes accelerate sentence-transformers

In [ ]:
# ── Cell 2: Load small chat model + embedder ──────────────────────────────────
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer
import torch

CHAT_MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"  # small + fast. Swap for 3B if you want it even faster.

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

print("Loading chat tokenizer...")
chat_tokenizer = AutoTokenizer.from_pretrained(CHAT_MODEL_ID)

print("Loading chat LLM...")
chat_llm = AutoModelForCausalLM.from_pretrained(
    CHAT_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
chat_llm.eval()

print("Loading embedder...")
embedder = SentenceTransformer("BAAI/bge-small-en-v1.5", device="cuda")

print("Models ready.")

In [ ]:
# ── Cell 3: Cleanup ───────────────────────────────────────────────────────────
import gc, subprocess, time, torch

def kill_everything():
    subprocess.run(["pkill", "-9", "-f", "uvicorn"], check=False)
    subprocess.run(["pkill", "-9", "-f", "ngrok"],   check=False)
    time.sleep(2)
    for name in ["chat_llm", "chat_tokenizer", "embedder", "thr", "_chunk_cache"]:
        if name in globals():
            del globals()[name]
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    print("Cleaned up.")

kill_everything()

In [ ]:
# ── Cell 4: FastAPI chat server with RAG ──────────────────────────────────────
import hashlib
import threading
import nest_asyncio
import numpy as np
import torch
import uvicorn
from typing import List
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from pyngrok import ngrok
from google.colab import userdata

nest_asyncio.apply()

assert "chat_llm"       in globals(), "Run Cell 2 first (chat_llm not loaded)"
assert "chat_tokenizer" in globals(), "Run Cell 2 first (chat_tokenizer not loaded)"
assert "embedder"       in globals(), "Run Cell 2 first (embedder not loaded)"

NGROK_AUTH_TOKEN = userdata.get("NGROK_AUTH_TOKEN")

app = FastAPI()
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

_chunk_cache = {}  # { md5(context): (chunks, embeddings) }


# ── RAG helpers ───────────────────────────────────────────────────────────────

def chunk_transcript(text: str, words_per_chunk: int = 400, overlap: int = 60) -> list:
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunks.append(" ".join(words[i:i + words_per_chunk]))
        i += words_per_chunk - overlap
    return chunks or [text]


def get_chunks_and_embs(context: str):
    key = hashlib.md5(context.encode()).hexdigest()
    cached = _chunk_cache.get(key)
    if cached:
        return cached
    chunks = chunk_transcript(context)
    embs = embedder.encode(chunks, convert_to_numpy=True, normalize_embeddings=True)
    _chunk_cache[key] = (chunks, embs)
    return chunks, embs


def retrieve(context: str, query: str, k: int = 10) -> str:
    chunks, embs = get_chunks_and_embs(context)
    q_emb = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True)[0]
    sims = embs @ q_emb
    top = np.argsort(-sims)[:k]
    top_sorted = sorted(top.tolist())  # preserve chronological order in prompt
    return "\n\n---\n\n".join(chunks[i] for i in top_sorted)


# ── Schema + route ────────────────────────────────────────────────────────────

class ChatMessage(BaseModel):
    role: str
    content: str


class ChatRequest(BaseModel):
    context: str
    messages: List[ChatMessage]


@app.post("/chat")
def chat(req: ChatRequest):
    last_user = next((m.content for m in reversed(req.messages) if m.role == "user"), "")
    relevant = retrieve(req.context, last_user, k=5) if last_user else req.context[:8000]

    system = (
        "You are a helpful assistant answering questions about a local government meeting. "
        "Use ONLY the meeting excerpts below. If the answer isn't in the excerpts, say you don't know.\n\n"
        f"MEETING EXCERPTS:\n{relevant}"
    )
    prompt = f"<|im_start|>system\n{system}<|im_end|>\n"
    for m in req.messages:
        role = m.role if m.role in ("user", "assistant") else "user"
        prompt += f"<|im_start|>{role}\n{m.content}<|im_end|>\n"
    prompt += "<|im_start|>assistant\n"

    inputs = chat_tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = chat_llm.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.3,
            do_sample=True,
            pad_token_id=chat_tokenizer.eos_token_id,
        )
    reply = chat_tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()
    return {"reply": reply}


@app.get("/health")
def health():
    return {"ok": True, "cached_transcripts": len(_chunk_cache)}


# ── start server ──────────────────────────────────────────────────────────────

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
public_url = ngrok.connect(8001)
print(f"Chat Ngrok URL: {public_url}", flush=True)

thr = threading.Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8001), daemon=True)
thr.start()